# 🎓 DeepfakeGuard — Fine-Tuning Wav2Vec2

Fine-tune `garystafford/wav2vec2-deepfake-voice-detector` on a voice deepfake detection dataset.

**Runtime → Change runtime type → T4 GPU** (required)

---

## 1. Setup & Installation

In [ ]:
!pip install -q transformers[torch] datasets evaluate accelerate
!pip install -q librosa soundfile scikit-learn tensorboard

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
from huggingface_hub import login

# Get token at: https://huggingface.co/settings/tokens
login(token="YOUR_HF_TOKEN_HERE")
print("✅ Logged in to Hugging Face")

## 2. Load Dataset

**Choose ONE of the options below.**

### Option A — ASVspoof 2021 (Recommended)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ntt-ds/asvspoof21", trust_remote_code=True)
print(f"Train: {len(dataset['train'])}, Validation: {len(dataset['dev'])}")
print(dataset)

### Option B — Your Own Dataset (Google Drive folders)

In [ ]:
import os
import librosa
import numpy as np
from datasets import Dataset, DatasetDict

def create_dataset_from_folders(real_folder, fake_folder, sr=16000, max_dur=10.0):
    samples = []
    exts = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
    max_samples = int(max_dur * sr)

    for label, folder in [(0, real_folder), (1, fake_folder)]:
        name = 'REAL' if label == 0 else 'FAKE'
        for f in sorted(os.listdir(folder)):
            if os.path.splitext(f)[1].lower() not in exts:
                continue
            try:
                audio, _ = librosa.load(os.path.join(folder, f), sr=sr, mono=True)
                if len(audio) > max_samples:
                    audio = audio[:max_samples]
                if len(audio) < sr:
                    continue
                samples.append({
                    'audio': {'array': audio, 'sampling_rate': sr},
                    'label': label,
                })
            except Exception as e:
                print(f'  ⚠️ {f}: {e}')
        print(f'{name}: {sum(1 for s in samples if s["label"] == label)} files loaded')

    ds = Dataset.from_list(samples)
    split = ds.train_test_split(test_size=0.2, seed=42, stratify_by_column='label')
    tv = split['test'].train_test_split(test_size=0.5, seed=42, stratify_by_column='label')
    return DatasetDict({'train': split['train'], 'validation': tv['train'], 'test': tv['test']})

from google.colab import drive
drive.mount('/content/drive')

REAL_PATH = "/content/drive/MyDrive/deepfake-dataset/real"   # ← Change this
FAKE_PATH = "/content/drive/MyDrive/deepfake-dataset/fake"   # ← Change this

dataset = create_dataset_from_folders(REAL_PATH, FAKE_PATH)
print(f'\nTrain: {len(dataset["train"])}, Val: {len(dataset["validation"])}, Test: {len(dataset["test"])}')

### Option C — Synthetic Data (Pipeline Testing Only)

In [ ]:
import numpy as np
from datasets import Dataset, DatasetDict

def synthetic_dataset(n=400, sr=16000, dur=4.0):
    samples = []
    for i in range(n // 2):
        t = np.linspace(0, dur, int(sr * dur))
        audio = (np.sin(2*np.pi*(150+np.random.randn()*20)*t)*0.3 + np.random.randn(len(t))*0.05).astype(np.float32)
        samples.append({'audio': {'array': audio, 'sampling_rate': sr}, 'label': 0})
    for i in range(n // 2):
        t = np.linspace(0, dur, int(sr * dur))
        audio = (np.sign(np.sin(2*np.pi*(120+np.random.randn()*15)*t))*0.3 + np.random.randn(len(t))*0.08).astype(np.float32)
        samples.append({'audio': {'array': audio, 'sampling_rate': sr}, 'label': 1})
    ds = Dataset.from_list(samples)
    sp = ds.train_test_split(test_size=0.2, seed=42, stratify_by_column='label')
    tv = sp['test'].train_test_split(test_size=0.5, seed=42, stratify_by_column='label')
    return DatasetDict({'train': sp['train'], 'validation': tv['train'], 'test': tv['test']})

dataset = synthetic_dataset()
print(f'Synthetic: Train={len(dataset["train"])}, Val={len(dataset["validation"])}')

## 3. Load Base Model

In [ ]:
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

MODEL_ID = "garystafford/wav2vec2-deepfake-voice-detector"

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    label2id={'bonafide': 0, 'spoof': 1},
    id2label={0: 'bonafide', 1: 'spoof'},
    ignore_mismatched_sizes=True,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"✅ Model: {MODEL_ID}")
print(f"   Labels: {model.config.label2id}")
print(f"   Device: {device}")
print(f"   Params: {sum(p.numel() for p in model.parameters()):,}")

## 4. Preprocess Audio

In [ ]:
SAMPLE_RATE = 16000
MAX_LENGTH = SAMPLE_RATE * 10  # 10 seconds

def preprocess(examples):
    audios = [x['array'] for x in examples['audio']]
    processed = []
    for a in audios:
        if len(a) > MAX_LENGTH:
            a = a[:MAX_LENGTH]
        elif len(a) < MAX_LENGTH:
            a = np.pad(a, (0, MAX_LENGTH - len(a)), mode='constant')
        processed.append(a)
    inputs = feature_extractor(processed, sampling_rate=SAMPLE_RATE, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LENGTH)
    inputs['labels'] = examples['label']
    return inputs

print("Preprocessing...")
tokenized_train = dataset['train'].map(preprocess, batched=True, batch_size=16, remove_columns=['audio'])
tokenized_val = dataset['validation'].map(preprocess, batched=True, batch_size=16, remove_columns=['audio'])
tokenized_test = dataset['test'].map(preprocess, batched=True, batch_size=16, remove_columns=['audio'])
print(f"✅ Train: {len(tokenized_train)}, Val: {len(tokenized_val)}, Test: {len(tokenized_test)}")

## 5. Fine-Tune

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

OUTPUT_DIR = "./deepfakeguard-finetuned"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    logging_steps=50,
    report_to="tensorboard",
    seed=42,
)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1': f1_metric.compute(predictions=preds, references=labels)['f1'],
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    tokenizer=feature_extractor,
)

import time
start = time.time()
train_result = trainer.train()
elapsed = (time.time() - start) / 60

print(f"\n✅ Training done in {elapsed:.1f} min")
print(f"   Loss: {train_result.metrics['train_loss']:.4f}")

eval_results = trainer.evaluate()
print(f"   Val Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"   Val F1:       {eval_results['eval_f1']:.4f}")

## 6. Evaluate on Test Set

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

preds_output = trainer.predict(tokenized_test)
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

print(classification_report(labels, preds, target_names=['Bonafide (Real)', 'Spoof (Fake)']))

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 7. Save & Download

In [ ]:
FINAL_DIR = "./deepfakeguard-final"
trainer.save_model(FINAL_DIR)
feature_extractor.save_pretrained(FINAL_DIR)

# Download as ZIP
import shutil
from google.colab import files
shutil.make_archive("deepfakeguard-finetuned", "zip", ".", FINAL_DIR)
files.download("deepfakeguard-finetuned.zip")
print(f"✅ Downloaded! Extract to C:\\deepfakeguard\\backend\\models\\ and update MODEL_ID in main.py")

In [ ]:
# Push to HuggingFace Hub (optional)
HF_REPO = "your-username/deepfakeguard-finetuned"  # ← Change this
model.push_to_hub(HF_REPO)
feature_extractor.push_to_hub(HF_REPO)
print(f"✅ Pushed to https://huggingface.co/{HF_REPO}")

## 8. Use in DeepfakeGuard Backend

In `backend/main.py`, change:
```python
MODEL_ID = "garystafford/wav2vec2-deepfake-voice-detector"
```
To:
```python
MODEL_ID = "./models/deepfakeguard-final"  # local path
# OR
MODEL_ID = "your-username/deepfakeguard-finetuned"  # HuggingFace
```

Then restart: `python main.py`